# 04 — Produce sample books

Run the **production pipeline** (settings from `configs/pipeline.toml`, frozen in notebook 03) on one small book per faculty and produce shareable sample PDFs in `output/samples/`.

This is the dress rehearsal for the batch run: five books, five faculties, different paper/print/scan quality.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import time

import pandas as pd

from evilflowers_books_digitalizer import BookContext, BookSource, LocalCache, load_settings
from evilflowers_books_digitalizer.pipeline.factory import build_pipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
for noisy in ("httpx", "ocrmypdf", "fontTools", "pikepdf"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

settings = load_settings()
cache = LocalCache(settings.cache_dir)
samples_dir = settings.output_dir / "samples"

## Select one book per faculty

Smallest book with ≥ 10 frames from each share (inventory cached by notebook 01). Adjust `MIN_FRAMES` / pick specific `BOOKS` entries to taste.

In [ ]:
MIN_FRAMES = 10

BOOKS = {}
for key in settings.sources:
    stats = cache.load_stats(key)
    if stats is None:
        raise RuntimeError("no cached inventory — run notebook 01 first")
    candidates = [b for b in stats.books if b.n_pages >= MIN_FRAMES]
    BOOKS[key] = min(candidates, key=lambda b: b.total_bytes)

pd.DataFrame(
    {"faculty": b.source, "book_id": b.book_id, "frames": b.n_pages, "gb": b.total_bytes / 1e9}
    for b in BOOKS.values()
).round(2)

## Run the production pipeline

Per book: download (resumable) → preprocess (split/crop/deskew) → assemble → OCR (PDF/A-2, `slk`) → enrich. Books are processed independently — a failure on one doesn't stop the rest.

In [ ]:
rows = []
for key, book in BOOKS.items():
    source = BookSource(settings.sources[key])
    pipeline = build_pipeline(source, cache, jobs=settings.ocr_jobs)
    ctx = BookContext(
        source=key,
        book_id=book.book_id,
        work_dir=cache.book_dir(key, book.book_id),
        output_dir=samples_dir,
    )
    started = time.perf_counter()
    try:
        ctx = pipeline.run(ctx)
        pdf = ctx.artifacts["pdf"]
        rows.append({
            "faculty": key,
            "book_id": book.book_id,
            "frames": ctx.metadata.get("n_frames"),
            "pages": ctx.metadata.get("n_pages"),
            "pdf_mb": pdf.stat().st_size / 1e6,
            "mb_per_page": pdf.stat().st_size / 1e6 / max(ctx.metadata.get("n_pages", 1), 1),
            "ocr_chars": ctx.metadata.get("n_text_chars"),
            "minutes": (time.perf_counter() - started) / 60,
            "pdf": str(pdf),
        })
    except Exception as exc:  # noqa: BLE001 — isolate per-book failures
        rows.append({"faculty": key, "book_id": book.book_id, "error": str(exc)[:200]})

report = pd.DataFrame(rows)
report.round(2)

## Review

First pages of every sample — covers and early pages are the most fragile part of the preprocessing (stamps, black bed, inserts).

In [ ]:
import cv2
import matplotlib.pyplot as plt
import pypdfium2 as pdfium

ok = [r for r in rows if "pdf" in r]
fig, axes = plt.subplots(len(ok), 4, figsize=(13, 4.2 * len(ok)))
for axrow, r in zip(axes.reshape(len(ok), 4), ok):
    doc = pdfium.PdfDocument(r["pdf"])
    for ax, page_no in zip(axrow, (0, 2, len(doc) // 2, len(doc) - 1)):
        img = doc[page_no].render(scale=0.35).to_numpy()
        ax.imshow(img)
        ax.set_title(f"{r['faculty']} p{page_no + 1}", fontsize=8)
        ax.axis("off")
fig.tight_layout()

In [ ]:
from pathlib import Path

for r in ok:
    text = Path(r["pdf"]).with_suffix(".txt").read_text(errors="ignore")
    snippet = " ".join(text[3000:3400].split())
    print(f"--- {r['faculty']} / {r['book_id'][:50]}\n{snippet}\n")

---
If a faculty's sample looks off (mis-splits, skew, washed-out print), take its book to **notebook 03** and tune `analyze_spread`/OCR parameters — then re-freeze `configs/pipeline.toml` and re-run this notebook.

**Next:** batch runner over all 880 books (resumable, per-book error isolation), then classification & embedding pipelines.